# Perception pipeline on a Kaggle driving clip

Attach a Kaggle dataset containing `best.pt` and a short BDD100K validation video clip (or a sample dashcam clip). Enable a GPU accelerator. The notebook runs detection, ByteTrack, classical lane detection, pinhole distance/TTC/risk, and writes a HUD overlay MP4 to `/kaggle/working`. MiDaS is disabled for this reproducible pass; the pipeline runs without network access.

In [ ]:
import subprocess
import sys
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("The video notebook requires a CUDA GPU. Enable a Kaggle GPU accelerator and rerun.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

REPO_URL = "https://github.com/30nikamvaishnavi/autonomous-driving-perception-stack.git"
REPO_DIR = Path("/kaggle/working/autonomous-driving-perception-stack")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml,export]"],
    cwd=REPO_DIR,
    check=True,
)
sys.path.insert(0, str(REPO_DIR / "src"))


In [ ]:
INPUT_DIR = Path("/kaggle/input")
BEST_CANDIDATES = sorted(INPUT_DIR.rglob("best.pt"))
VIDEO_CANDIDATES = sorted(INPUT_DIR.rglob("*.mp4"))
if not BEST_CANDIDATES:
    raise FileNotFoundError("No best.pt found under /kaggle/input; attach the trained model dataset.")
if not VIDEO_CANDIDATES:
    raise FileNotFoundError(
        "No MP4 found under /kaggle/input; attach a short BDD100K validation or dashcam video dataset."
    )
BEST_PT = BEST_CANDIDATES[0]
VIDEO_PATH = VIDEO_CANDIDATES[0]
MAX_FRAMES = 300
OUTPUT_VIDEO = Path("/kaggle/working/pipeline_hud.mp4")
print(f"Weights: {BEST_PT}")
print(f"Input video: {VIDEO_PATH}")


In [ ]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "adpstack.cli",
        "run",
        "--source",
        str(VIDEO_PATH),
        "--weights",
        str(BEST_PT),
        "--out",
        str(OUTPUT_VIDEO),
        "--imgsz",
        "640",
        "--device",
        "0",
        "--no-depth",
        "--max-frames",
        str(MAX_FRAMES),
    ],
    cwd=REPO_DIR,
    check=True,
)
if not OUTPUT_VIDEO.is_file() or OUTPUT_VIDEO.stat().st_size == 0:
    raise RuntimeError(f"Pipeline did not produce a non-empty output video: {OUTPUT_VIDEO}")
print(f"HUD video: {OUTPUT_VIDEO}")
